# EMG2QWERTY: Experiment Results Analysis

**ECE C147/C247 — Neural Networks & Deep Learning, Winter 2026**  
**UCLA ECE — Prof. J.C. Kao**

This notebook consolidates experimental results across all team members,
loads training logs from the GCP instance, and generates publication-ready
figures for the final project report.

**Experiments covered:**
1. TDS-Conv baseline (40 epochs)
2. Bidirectional GRU (40 and 150 epochs)
3. Bidirectional LSTM (150 epochs)
4. GRU+CNN hybrid (≈129 epochs)

---
## 1. Setup & NeurIPS-Style Configuration

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ──────────────────────────────────────────────
# NeurIPS 2024 style: two-column, serif fonts
# ──────────────────────────────────────────────
SINGLE_COL = 3.25   # inches — single column width
DOUBLE_COL = 6.75   # inches — full (two-column) width
GOLDEN     = 1.618  # golden ratio for height

plt.rcParams.update({
    # Fonts – match NeurIPS template (Times / serif)
    'font.family':       'serif',
    'font.serif':        ['Times New Roman', 'Times', 'DejaVu Serif'],
    'mathtext.fontset':  'stix',
    'font.size':         8,
    'axes.titlesize':    9,
    'axes.labelsize':    8,
    'xtick.labelsize':   7,
    'ytick.labelsize':   7,
    'legend.fontsize':   7,
    'figure.titlesize':  10,

    # Figure defaults
    'figure.figsize':    (DOUBLE_COL, DOUBLE_COL / GOLDEN),
    'figure.dpi':        150,
    'savefig.dpi':       300,
    'savefig.bbox':      'tight',
    'savefig.pad_inches': 0.02,

    # Axes
    'axes.linewidth':    0.6,
    'axes.grid':         True,
    'grid.alpha':        0.3,
    'grid.linewidth':    0.4,

    # Ticks
    'xtick.major.width': 0.6,
    'ytick.major.width': 0.6,
    'xtick.direction':   'in',
    'ytick.direction':   'in',

    # Lines
    'lines.linewidth':   1.2,
    'lines.markersize':  4,

    # Legend
    'legend.frameon':       True,
    'legend.framealpha':    0.9,
    'legend.edgecolor':     '0.8',
    'legend.borderaxespad': 0.5,
})

# Color palette — colorblind-friendly
COLORS = {
    'TDS-Conv (baseline)': '#0072B2',  # blue
    'GRU (40 ep)':         '#D55E00',  # vermillion
    'GRU (150 ep)':        '#E69F00',  # orange
    'LSTM (150 ep)':       '#009E73',  # green
    'GRU+CNN':             '#CC79A7',  # pink
}

MODEL_ORDER = [
    'TDS-Conv (baseline)',
    'GRU (40 ep)',
    'GRU (150 ep)',
    'LSTM (150 ep)',
    'GRU+CNN',
]

# Output directory for saved figures
FIG_DIR = Path('figures')
FIG_DIR.mkdir(exist_ok=True)

print('Setup complete.')

---
## 2. Experiment Configuration & GCP Log Paths

Each experiment was run by a different team member on the shared GCP instance.
Update the paths below if logs are moved.

In [ ]:
# ──────────────────────────────────────────────────────────
# Experiment registry — edit paths here if anything moves
# ──────────────────────────────────────────────────────────
EXPERIMENTS = {
    'TDS-Conv (baseline)': {
        'user':       'christiansolano',
        'log_dir':    '/home/christiansolano/emg2qwerty/logs/2026-03-09/01-53-48',
        'epochs':     40,
        'arch':       'TDS-Conv',
        'description': 'Baseline TDS-Conv encoder with 4 blocks (24 ch each), kernel width 32',
    },
    'GRU (40 ep)': {
        'user':       'christiansolano',
        'log_dir':    '/home/christiansolano/emg2qwerty/logs/2026-03-09/04-19-13',
        'epochs':     40,
        'arch':       'GRU',
        'description': 'Bidirectional GRU (2 layers, hidden=384), early stopping at ~40 epochs',
    },
    'GRU (150 ep)': {
        'user':       'christiansolano',
        'log_dir':    '/home/christiansolano/emg2qwerty/logs/2026-03-09/04-23-38',
        'epochs':     150,
        'arch':       'GRU',
        'description': 'Bidirectional GRU (2 layers, hidden=384), full 150 epoch training',
    },
    'LSTM (150 ep)': {
        'user':       'jameshall',
        'log_dir':    '/home/jameshall/emg2qwerty/logs/2026-03-11/18-40-23',
        'epochs':     150,
        'arch':       'LSTM',
        'description': 'Bidirectional LSTM (2 layers, hidden=384), full 150 epoch training',
    },
    'GRU+CNN': {
        'user':       'lucas',
        'log_dir':    '/home/lucas/emg2qwerty/logs/2026-03-12/00-25-03',
        'epochs':     129,
        'arch':       'GRU+CNN',
        'description': 'GRU+CNN hybrid encoder',
    },
}

print(f'Registered {len(EXPERIMENTS)} experiments:')
for name, cfg in EXPERIMENTS.items():
    print(f'  • {name:25s}  ({cfg["user"]})')

---
## 3. Load Training Logs (CSV)

PyTorch Lightning's CSVLogger writes a `metrics.csv` file in each run's
`lightning_logs/version_*/` directory.  We parse these to get epoch-by-epoch
training and validation curves.

In [ ]:
def find_metrics_csv(log_dir: str) -> str | None:
    """Search for metrics.csv inside a Lightning log directory.
    
    Looks in common locations:
      <log_dir>/lightning_logs/version_*/metrics.csv
      <log_dir>/csv_logs/version_*/metrics.csv
      <log_dir>/metrics.csv
    """
    search_patterns = [
        os.path.join(log_dir, 'lightning_logs', 'version_*', 'metrics.csv'),
        os.path.join(log_dir, 'csv_logs', 'version_*', 'metrics.csv'),
        os.path.join(log_dir, 'metrics.csv'),
        os.path.join(log_dir, '**', 'metrics.csv'),
    ]
    for pattern in search_patterns:
        matches = sorted(glob.glob(pattern, recursive=True))
        if matches:
            return matches[-1]  # latest version
    return None


def load_training_log(csv_path: str) -> pd.DataFrame:
    """Load and clean a Lightning metrics.csv file.
    
    Lightning logs train and val metrics on separate rows.
    This function merges them by epoch for easy plotting.
    """
    df = pd.read_csv(csv_path)
    
    # Identify epoch column
    epoch_col = 'epoch'
    if epoch_col not in df.columns:
        print(f'  Warning: no epoch column in {csv_path}')
        return df
    
    # Group by epoch, taking the last non-NaN value for each metric
    grouped = df.groupby(epoch_col).last().reset_index()
    
    # Forward-fill val metrics (logged less frequently than train)
    val_cols = [c for c in grouped.columns if c.startswith('val')]
    grouped[val_cols] = grouped[val_cols].ffill()
    
    return grouped


# ── Load all available logs ──
training_logs = {}
for name, cfg in EXPERIMENTS.items():
    csv_path = find_metrics_csv(cfg['log_dir'])
    if csv_path is not None:
        training_logs[name] = load_training_log(csv_path)
        n_epochs = len(training_logs[name])
        print(f'  ✓ {name:25s}  →  {n_epochs} epochs loaded from {csv_path}')
    else:
        print(f'  ✗ {name:25s}  →  metrics.csv NOT FOUND in {cfg["log_dir"]}')
        print(f'    Searched: lightning_logs/version_*/metrics.csv and alternatives')

print(f'\nLoaded training logs for {len(training_logs)}/{len(EXPERIMENTS)} experiments.')

---
## 4. Final Results Summary

Hardcoded from the experiment outputs (validated against checkpoints).
These serve as ground truth even if CSV logs are incomplete.

In [ ]:
# ── Final evaluation metrics (from experiment results document) ──
FINAL_RESULTS = pd.DataFrame([
    {
        'Model':     'TDS-Conv (baseline)',
        'Epochs':    40,
        'val/CER':   22.3748, 'val/DER':   1.8830, 'val/IER':   6.2029, 'val/SER':  14.2889,  'val/loss':  0.7399,
        'test/CER':  24.4003, 'test/DER':  1.7722, 'test/IER':  5.2950, 'test/SER': 17.3330,  'test/loss': 0.7722,
        'best_epoch': 39,
    },
    {
        'Model':     'GRU (40 ep)',
        'Epochs':    40,
        'val/CER':   22.9951, 'val/DER':   2.5255, 'val/IER':   7.1777, 'val/SER':  13.2920,  'val/loss':  0.7590,
        'test/CER':  23.4061, 'test/DER':  2.3125, 'test/IER':  5.1437, 'test/SER': 15.9499,  'test/loss': 0.7399,
        'best_epoch': None,  # checkpoint not saved
    },
    {
        'Model':     'GRU (150 ep)',
        'Epochs':    150,
        'val/CER':   15.1086, 'val/DER':   1.6172, 'val/IER':   3.6553, 'val/SER':   9.8361,  'val/loss':  0.6397,
        'test/CER':  15.9715, 'test/DER':  0.9726, 'test/IER':  4.0415, 'test/SER': 10.9574,  'test/loss': 0.6409,
        'best_epoch': 111,
    },
    {
        'Model':     'LSTM (150 ep)',
        'Epochs':    150,
        'val/CER':   13.1812, 'val/DER':   1.3735, 'val/IER':   2.8799, 'val/SER':   8.9278,  'val/loss':  0.6127,
        'test/CER':  14.2857, 'test/DER':  1.1671, 'test/IER':  2.4854, 'test/SER': 10.6332,  'test/loss': 0.6211,
        'best_epoch': 145,
    },
    {
        'Model':     'GRU+CNN',
        'Epochs':    129,
        'val/CER':   15.0194, 'val/DER':   1.5507, 'val/IER':   3.9211, 'val/SER':   9.5481,  'val/loss':  0.6984,
        'test/CER':  15.9066, 'test/DER':  0.6051, 'test/IER':  4.9708, 'test/SER': 10.3307,  'test/loss': 0.6705,
        'best_epoch': 128,
    },
])

FINAL_RESULTS = FINAL_RESULTS.set_index('Model')

# Display a clean summary table
display_cols = ['Epochs', 'val/CER', 'val/loss', 'test/CER', 'test/loss', 'best_epoch']
styled = (
    FINAL_RESULTS[display_cols]
    .style
    .format({
        'val/CER':  '{:.2f}', 'val/loss':  '{:.4f}',
        'test/CER': '{:.2f}', 'test/loss': '{:.4f}',
    })
    .highlight_min(subset=['test/CER', 'test/loss'], color='#d4edda')
    .set_caption('Table 1: Final evaluation metrics across all architectures')
)
display(styled)

In [ ]:
# ── Full results table with error-type breakdown ──
full_display = ['Epochs', 'test/CER', 'test/DER', 'test/IER', 'test/SER', 'test/loss']
styled_full = (
    FINAL_RESULTS[full_display]
    .style
    .format({
        'test/CER': '{:.2f}', 'test/DER': '{:.2f}',
        'test/IER': '{:.2f}', 'test/SER': '{:.2f}',
        'test/loss': '{:.4f}',
    })
    .highlight_min(subset=['test/CER'], color='#d4edda')
    .set_caption('Table 2: Test set error breakdown — CER = DER + IER + SER')
)
display(styled_full)

---
## 5. Training Curves

Loss and CER over training epochs for all models that have CSV logs available.

In [ ]:
def plot_training_curves(metric: str, ylabel: str, title: str,
                         figsize=None, savename=None):
    """Plot a training metric across all experiments with available logs."""
    if figsize is None:
        figsize = (DOUBLE_COL, DOUBLE_COL / GOLDEN)
    
    fig, axes = plt.subplots(1, 2, figsize=figsize, sharey=True)
    
    for name in MODEL_ORDER:
        if name not in training_logs:
            continue
        df = training_logs[name]
        color = COLORS[name]
        
        # Training metric
        train_col = f'train/{metric}'
        val_col   = f'val/{metric}'
        
        if train_col in df.columns:
            data = df[['epoch', train_col]].dropna()
            axes[0].plot(data['epoch'], data[train_col],
                        color=color, label=name, alpha=0.85)
        
        if val_col in df.columns:
            data = df[['epoch', val_col]].dropna()
            axes[1].plot(data['epoch'], data[val_col],
                        color=color, label=name, alpha=0.85)
    
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel(ylabel)
    axes[0].set_title('Training')
    
    axes[1].set_xlabel('Epoch')
    axes[1].set_title('Validation')
    axes[1].legend(loc='upper right')
    
    fig.suptitle(title, y=1.02)
    fig.tight_layout()
    
    if savename:
        fig.savefig(FIG_DIR / savename)
        print(f'Saved: {FIG_DIR / savename}')
    
    plt.show()


if training_logs:
    # List available columns from the first log to understand structure
    sample_name = list(training_logs.keys())[0]
    print(f'Available columns in {sample_name}:')
    print(training_logs[sample_name].columns.tolist())
else:
    print('No training logs loaded — skipping curve plots.')
    print('Run this notebook on the GCP instance to access log files.')

In [ ]:
# ── Plot loss curves ──
if training_logs:
    plot_training_curves(
        metric='loss',
        ylabel='CTC Loss',
        title='Figure 1: Training and Validation Loss',
        savename='loss_curves.pdf',
    )

In [ ]:
# ── Plot CER curves ──
if training_logs:
    plot_training_curves(
        metric='CER',
        ylabel='Character Error Rate (%)',
        title='Figure 2: Training and Validation CER',
        savename='cer_curves.pdf',
    )

---
## 6. Architecture Comparison

Side-by-side comparison of test CER across all architectures.

In [ ]:
# ── Bar chart: Test CER comparison ──
fig, ax = plt.subplots(figsize=(DOUBLE_COL, SINGLE_COL * 1.1))

models = [m for m in MODEL_ORDER if m in FINAL_RESULTS.index]
test_cers = FINAL_RESULTS.loc[models, 'test/CER'].values
val_cers  = FINAL_RESULTS.loc[models, 'val/CER'].values
colors    = [COLORS[m] for m in models]

x = np.arange(len(models))
width = 0.35

bars_val  = ax.bar(x - width/2, val_cers,  width, color=colors, alpha=0.5,
                   edgecolor=[c for c in colors], linewidth=0.8, label='Validation')
bars_test = ax.bar(x + width/2, test_cers, width, color=colors, alpha=0.9,
                   edgecolor='black', linewidth=0.5, label='Test')

# Add value labels on bars
for bar in bars_test:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.3, f'{h:.1f}',
            ha='center', va='bottom', fontsize=7, fontweight='bold')

ax.set_ylabel('Character Error Rate (%)')
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=15, ha='right')
ax.legend()
ax.set_title('Figure 3: Architecture Comparison — Validation and Test CER')
ax.set_ylim(0, max(test_cers) * 1.15)

fig.tight_layout()
fig.savefig(FIG_DIR / 'architecture_comparison.pdf')
print(f'Saved: {FIG_DIR / "architecture_comparison.pdf"}')
plt.show()

In [ ]:
# ── Improvement over baseline ──
baseline_cer = FINAL_RESULTS.loc['TDS-Conv (baseline)', 'test/CER']

print('Relative improvement over TDS-Conv baseline (test CER):')
print('=' * 55)
for model in models:
    cer = FINAL_RESULTS.loc[model, 'test/CER']
    improvement = (baseline_cer - cer) / baseline_cer * 100
    arrow = '↓' if improvement > 0 else '↑'
    print(f'  {model:25s}  CER={cer:6.2f}  ({arrow} {abs(improvement):5.1f}%)')

---
## 7. Error Type Breakdown

CER decomposes into three error types:  
- **SER** (Substitution Error Rate) — wrong character predicted  
- **IER** (Insertion Error Rate) — extra characters predicted  
- **DER** (Deletion Error Rate) — characters missed  

CER = SER + IER + DER

In [ ]:
# ── Stacked bar chart: Error type breakdown ──
fig, ax = plt.subplots(figsize=(DOUBLE_COL, SINGLE_COL * 1.2))

models = [m for m in MODEL_ORDER if m in FINAL_RESULTS.index]
x = np.arange(len(models))

ser_vals = FINAL_RESULTS.loc[models, 'test/SER'].values
ier_vals = FINAL_RESULTS.loc[models, 'test/IER'].values
der_vals = FINAL_RESULTS.loc[models, 'test/DER'].values

bar_width = 0.55

p1 = ax.bar(x, ser_vals, bar_width, label='SER (Substitution)', color='#4e79a7', edgecolor='white', linewidth=0.5)
p2 = ax.bar(x, ier_vals, bar_width, bottom=ser_vals, label='IER (Insertion)', color='#f28e2b', edgecolor='white', linewidth=0.5)
p3 = ax.bar(x, der_vals, bar_width, bottom=ser_vals + ier_vals, label='DER (Deletion)', color='#e15759', edgecolor='white', linewidth=0.5)

# Total CER labels on top
totals = ser_vals + ier_vals + der_vals
for i, total in enumerate(totals):
    ax.text(i, total + 0.3, f'{total:.1f}', ha='center', va='bottom',
            fontsize=7, fontweight='bold')

ax.set_ylabel('Error Rate (%)')
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=15, ha='right')
ax.legend(loc='upper right')
ax.set_title('Figure 4: Test CER Decomposition by Error Type')
ax.set_ylim(0, max(totals) * 1.15)

fig.tight_layout()
fig.savefig(FIG_DIR / 'error_breakdown.pdf')
print(f'Saved: {FIG_DIR / "error_breakdown.pdf"}')
plt.show()

In [ ]:
# ── Error type proportions ──
fig, axes = plt.subplots(1, len(models), figsize=(DOUBLE_COL, SINGLE_COL * 0.9))

error_colors = ['#4e79a7', '#f28e2b', '#e15759']
error_labels = ['SER', 'IER', 'DER']

for i, model in enumerate(models):
    ser = FINAL_RESULTS.loc[model, 'test/SER']
    ier = FINAL_RESULTS.loc[model, 'test/IER']
    der = FINAL_RESULTS.loc[model, 'test/DER']
    vals = [ser, ier, der]
    
    axes[i].pie(vals, colors=error_colors, autopct='%1.0f%%',
                textprops={'fontsize': 6}, startangle=90,
                pctdistance=0.7)
    axes[i].set_title(model, fontsize=7, pad=4)

# Common legend
fig.legend(error_labels, loc='lower center', ncol=3, fontsize=7,
           bbox_to_anchor=(0.5, -0.05))
fig.suptitle('Figure 5: Error Type Proportions per Architecture', y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / 'error_proportions.pdf')
print(f'Saved: {FIG_DIR / "error_proportions.pdf"}')
plt.show()

---
## 8. Val vs Test Gap Analysis

Examining the generalization gap (test CER − val CER) across architectures
can reveal overfitting tendencies.

In [ ]:
# ── Generalization gap ──
fig, ax = plt.subplots(figsize=(SINGLE_COL * 1.8, SINGLE_COL * 1.1))

models = [m for m in MODEL_ORDER if m in FINAL_RESULTS.index]
gaps = (FINAL_RESULTS.loc[models, 'test/CER'] - FINAL_RESULTS.loc[models, 'val/CER']).values
colors_list = [COLORS[m] for m in models]

bars = ax.barh(range(len(models)), gaps, color=colors_list, edgecolor='black', linewidth=0.5)

for i, (gap, bar) in enumerate(zip(gaps, bars)):
    ax.text(gap + 0.1, i, f'{gap:+.2f}', va='center', fontsize=7)

ax.set_yticks(range(len(models)))
ax.set_yticklabels(models)
ax.set_xlabel('Test CER − Val CER (%)')
ax.set_title('Figure 6: Generalization Gap')
ax.axvline(x=0, color='black', linewidth=0.5, linestyle='--')

fig.tight_layout()
fig.savefig(FIG_DIR / 'generalization_gap.pdf')
print(f'Saved: {FIG_DIR / "generalization_gap.pdf"}')
plt.show()

---
## 9. Effect of Training Duration

Direct comparison of the GRU at 40 vs 150 epochs to quantify the impact
of extended training.

In [ ]:
# ── Training duration impact (GRU 40 vs 150 epochs) ──
gru_short = FINAL_RESULTS.loc['GRU (40 ep)']
gru_long  = FINAL_RESULTS.loc['GRU (150 ep)']

metrics_compare = ['test/CER', 'test/DER', 'test/IER', 'test/SER', 'test/loss']
labels = ['CER', 'DER', 'IER', 'SER', 'Loss']

fig, ax = plt.subplots(figsize=(SINGLE_COL * 1.6, SINGLE_COL))

x = np.arange(len(labels))
width = 0.3

vals_short = [gru_short[m] for m in metrics_compare]
vals_long  = [gru_long[m]  for m in metrics_compare]

ax.bar(x - width/2, vals_short, width, color=COLORS['GRU (40 ep)'],
       label='GRU (40 epochs)', edgecolor='black', linewidth=0.5)
ax.bar(x + width/2, vals_long,  width, color=COLORS['GRU (150 ep)'],
       label='GRU (150 epochs)', edgecolor='black', linewidth=0.5)

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Value')
ax.legend(fontsize=6)
ax.set_title('Figure 7: Effect of Training Duration (GRU)')

fig.tight_layout()
fig.savefig(FIG_DIR / 'training_duration.pdf')
print(f'Saved: {FIG_DIR / "training_duration.pdf"}')
plt.show()

# Print improvement
cer_improve = (gru_short['test/CER'] - gru_long['test/CER']) / gru_short['test/CER'] * 100
print(f'\nExtending GRU training from 40 to 150 epochs:')
print(f'  Test CER: {gru_short["test/CER"]:.2f} → {gru_long["test/CER"]:.2f}  ({cer_improve:.1f}% reduction)')

---
## 10. Summary & Discussion

In [ ]:
# ── Programmatic summary statistics ──
best_model = FINAL_RESULTS['test/CER'].idxmin()
best_cer   = FINAL_RESULTS.loc[best_model, 'test/CER']
worst_model = FINAL_RESULTS['test/CER'].idxmax()
worst_cer   = FINAL_RESULTS.loc[worst_model, 'test/CER']

print('══════════════════════════════════════════')
print('          EXPERIMENT SUMMARY')
print('══════════════════════════════════════════')
print(f'  Best model:   {best_model} (test CER = {best_cer:.2f}%)')
print(f'  Worst model:  {worst_model} (test CER = {worst_cer:.2f}%)')
print(f'  Improvement:  {worst_cer - best_cer:.2f} percentage points')
print(f'                ({(worst_cer - best_cer)/worst_cer*100:.1f}% relative reduction)')
print()
print('Key findings:')
print(f'  1. LSTM achieves the lowest test CER ({best_cer:.2f}%), outperforming all other architectures.')
print(f'  2. Extended training (150 vs 40 epochs) reduced GRU test CER by {cer_improve:.1f}%.')
print(f'  3. GRU+CNN hybrid is competitive with GRU (150 ep) at similar CER levels.')
print(f'  4. Substitution errors (SER) dominate across all models ({"SER is largest component"}).')
print(f'  5. GRU+CNN achieves the lowest deletion error rate (DER = {FINAL_RESULTS.loc["GRU+CNN", "test/DER"]:.2f}%).')

### Key Observations

**Architecture comparison:** The bidirectional LSTM encoder achieved the best overall
performance with a test CER of ~14.3%, representing a substantial improvement over the
TDS-Conv baseline (~24.4%). Both recurrent architectures (GRU, LSTM) significantly
outperform the convolutional baseline when given sufficient training time, suggesting
that the sequential nature of sEMG signals benefits from architectures with explicit
temporal memory.

**Training duration matters:** The GRU comparison at 40 vs 150 epochs demonstrates
that these models continue to improve well beyond typical early-stopping thresholds.
The 40-epoch GRU barely outperforms the TDS-Conv baseline, while the 150-epoch
variant achieves dramatically lower CER — highlighting the importance of sufficient
training for recurrent architectures on this task.

**Error type analysis:** Substitution errors are the dominant error type across all
architectures, suggesting that discriminating between similar characters (which produce
similar muscle activation patterns) remains the core challenge. The GRU+CNN hybrid
achieves notably low deletion errors, potentially because the CNN component helps
preserve local temporal features that prevent character omissions.

**Generalization gap:** All models show a positive gap (test CER > val CER), which is
expected given the test set uses full sessions without windowing. The TDS-Conv baseline
shows the largest gap, while the recurrent models generalize more consistently.

---

*Notebook generated for ECE C147/C247 Winter 2026 Final Project.*  
*All figures saved to `figures/` directory in PDF format for direct inclusion in the NeurIPS-style report.*